# Practical work — Understanding LLM inference

**Duration:** about 30 minutes  
**Hardware:** one NVIDIA GB200 node  
**Inference engine:** vLLM, already running  
**Model:** one preloaded instruction-tuned model exposed through an OpenAI-compatible API

## Learning objectives

By the end of this practical work, you should be able to:

1. distinguish greedy decoding from sampling;
2. explain why prompt length mainly affects the time to the first token (TTFT);
3. explain why generating more tokens increases response time;
4. observe the trade-off between per-request latency and system throughput under concurrency;
5. estimate the memory occupied by model weights at different precisions.

## How to work

This is an observation-and-experiment notebook, not a programming exercise. The helper functions are provided. Your job is to **change a few parameters, run the cells, inspect the measurements and answer the questions**.

| Activity | Time |
|---|---:|
| Connect and warm up | 3 min |
| 1. Generation strategies | 5 min |
| 2. Prompt length and output length | 8 min |
| 3. Multiple concurrent users | 9 min |
| 4. Memory and quantization | 3 min |
| Wrap-up | 2 min |

> **Important:** results depend on the model, vLLM version, GB200 configuration and other requests using the server. Your measurements are observations from this run, not universal performance numbers.

## Before starting: the inference server

The instructor starts vLLM before the session and shares the API address. The model should already be loaded and warmed up. Students should not need to install packages or download model weights.

For per-request TTFT / generation metrics, the server should be started with `--enable-per-request-metrics` (supported by current vLLM releases). A typical command is:

```bash
vllm serve /path/to/preloaded-model \\
  --served-model-name course-llm \\
  --host 0.0.0.0 --port 8000 \\
  --enable-per-request-metrics
```

Adjust the model path, hostname, port and resource options to the cluster environment. Do not ask every student to start a separate server on the same GPU.

**Instructor setup:** make `openai`, `pandas` and `matplotlib` available in the notebook environment. If the notebook server runs on a different machine, set `LLM_API_BASE_URL` to the address reachable from that machine, for example `http://gb200-node:8000/v1`.

## 0. Connect to the model (3 min)

We use the OpenAI Python client because vLLM exposes an OpenAI-compatible API. The server handles model loading, tokenization, prefill, decode and KV-cache management.

Run the next cell. It discovers the served model name automatically unless `LLM_MODEL_NAME` is set in the environment.

In [ ]:
import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from openai import OpenAI

# Set these environment variables before launching Jupyter, or edit the defaults.
API_BASE_URL = os.getenv("LLM_API_BASE_URL", "http://localhost:8000/v1")
API_KEY = os.getenv("LLM_API_KEY", os.getenv("OPENAI_API_KEY", "EMPTY"))
MODEL_OVERRIDE = os.getenv("LLM_MODEL_NAME")  # optional

client = OpenAI(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    timeout=180.0,
    max_retries=0,
)

served_models = client.models.list().data
if not served_models:
    raise RuntimeError("The server is reachable but did not return any served model.")

MODEL = MODEL_OVERRIDE or served_models[0].id
print(f"API:   {API_BASE_URL}")
print(f"Model: {MODEL}")

In [ ]:
# One short warm-up request. This avoids including first-use effects in later measurements.
warmup = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is a language model?"}],
    temperature=0,
    max_tokens=40,
)
print(warmup.choices[0].message.content)
print("Warm-up complete.")

**Think:** what stages does a request go through between the client sending a prompt and receiving text?

*Hint: chat template and tokenization → prefill → first-token selection → autoregressive decode → detokenization / response.*

## Helper functions (provided)

The next cell contains the small amount of shared code used by the experiments. You do not need to modify it.

- `ask(...)` sends one chat request and records client-observed latency, token counts and vLLM per-request metrics when available.
- `run_load_test(...)` sends a fixed number of requests with a chosen maximum concurrency.

The client-observed latency includes network and client overhead. Server-side TTFT and ITL are reported separately when the server exposes them.

In [ ]:
def _get_server_metrics(response):
    """Read vLLM's optional per-request metrics from an OpenAI SDK response."""
    extra = getattr(response, "model_extra", None) or {}
    if not extra and hasattr(response, "model_dump"):
        extra = response.model_dump()
    return extra.get("metrics") or {}


def ask(prompt, *, temperature=0.0, top_p=1.0, max_tokens=64, _client=None):
    """Send one request and return its answer plus comparable measurements."""
    api = _client or client
    start = time.perf_counter()
    response = api.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    client_latency_ms = (time.perf_counter() - start) * 1000

    usage = response.usage
    metrics = _get_server_metrics(response)
    output_tokens = getattr(usage, "completion_tokens", None) if usage else None

    # vLLM reports these fields when --enable-per-request-metrics is active.
    ttft_ms = metrics.get("time_to_first_token_ms")
    mean_itl_ms = metrics.get("mean_itl_ms")
    generation_time_ms = metrics.get("generation_time_ms")

    return {
        "answer": response.choices[0].message.content or "",
        "input_tokens": getattr(usage, "prompt_tokens", None) if usage else None,
        "output_tokens": output_tokens,
        "client_latency_ms": client_latency_ms,
        "ttft_ms": ttft_ms,
        "mean_itl_ms": mean_itl_ms,
        "generation_time_ms": generation_time_ms,
        "queue_time_ms": metrics.get("queue_time_ms"),
    }


_thread_local = threading.local()

def _thread_client():
    """Create one reusable API client per worker thread."""
    if not hasattr(_thread_local, "client"):
        _thread_local.client = OpenAI(
            base_url=API_BASE_URL,
            api_key=API_KEY,
            timeout=180.0,
            max_retries=0,
        )
    return _thread_local.client


def run_load_test(num_requests=16, concurrency=1, max_tokens=80):
    """Send requests with at most `concurrency` requests in flight."""
    def worker(i):
        prompt = (
            f"Request {i + 1}. Explain in a clear, useful way why inference servers "
            "use a KV cache when generating a long answer. Give a few concrete details."
        )
        result = ask(
            prompt,
            temperature=0,
            max_tokens=max_tokens,
            _client=_thread_client(),
        )
        result["request_id"] = i + 1
        return result

    wall_start = time.perf_counter()
    rows = []
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = [pool.submit(worker, i) for i in range(num_requests)]
        for future in as_completed(futures):
            rows.append(future.result())
    wall_time_s = time.perf_counter() - wall_start

    df = pd.DataFrame(rows).sort_values("request_id").reset_index(drop=True)
    successful_output_tokens = df["output_tokens"].fillna(0).sum()
    summary = {
        "requests": len(df),
        "concurrency": concurrency,
        "wall_time_s": wall_time_s,
        "request_throughput_rps": len(df) / wall_time_s,
        "output_tokens_per_s": successful_output_tokens / wall_time_s,
        "p50_client_latency_s": df["client_latency_ms"].median() / 1000,
        "p95_client_latency_s": df["client_latency_ms"].quantile(0.95) / 1000,
        "p50_ttft_ms": df["ttft_ms"].median() if df["ttft_ms"].notna().any() else None,
        "p95_ttft_ms": df["ttft_ms"].quantile(0.95) if df["ttft_ms"].notna().any() else None,
        "p50_mean_itl_ms": df["mean_itl_ms"].median() if df["mean_itl_ms"].notna().any() else None,
    }
    return df, summary

print("Helpers ready.")

---

## 1. Generation strategies (5 min)

**Objective:** observe that the decoding strategy changes how the next token is selected, without changing the model weights.

We will use the same prompt with three configurations:

- **Greedy:** `temperature=0` (deterministic argmax selection in this API).
- **Conservative sampling:** low temperature.
- **More diverse sampling:** higher temperature and nucleus filtering with `top_p`.

Run each configuration twice. Compare the answers rather than expecting a particular sentence.

In [ ]:
PROMPT = "The capital of France is"

configs = [
    {"name": "Greedy", "temperature": 0.0, "top_p": 1.0},
    {"name": "Conservative sampling", "temperature": 0.3, "top_p": 1.0},
    {"name": "More diverse sampling", "temperature": 1.0, "top_p": 0.9},
]

sampling_rows = []
for config in configs:
    for repetition in range(2):
        result = ask(
            PROMPT,
            temperature=config["temperature"],
            top_p=config["top_p"],
            max_tokens=32,
        )
        sampling_rows.append({
            "configuration": config["name"],
            "run": repetition + 1,
            "answer": result["answer"].strip(),
        })

sampling_df = pd.DataFrame(sampling_rows)
display(sampling_df)

### Questions — generation

1. Were the greedy answers identical across repetitions? Were sampled answers identical?
2. What happens to the range of possible outputs when temperature increases?
3. Does a more diverse answer mean a more factually accurate answer?
4. Which setting would you choose for a factual question? Which for brainstorming?

**Key idea:** temperature reshapes the probability distribution before sampling. Greedy decoding always chooses the highest-scoring next token. Sampling can select a lower-probability candidate. Diversity is not a measure of correctness.

*Optional 30-second variation:* replace the prompt with `Suggest three original names for a small café in Paris.` and rerun the three configurations.

---

## 2. Prompt length vs output length (8 min)

**Objective:** distinguish the work done before the first generated token from the sequential work of generating subsequent tokens.

### 2A. Increase prompt length — observe prefill / TTFT

We keep the task and output budget fixed, but give the model more context. The notes are repeated to create short, medium and long prompts. The **actual token count** is measured by the serving model and shown in the results.

A longer prompt generally requires more prefill work before the first output token can be produced.

In [ ]:
NOTES = [
    "Paris is the capital of France and is located in the north-central part of the country. "
    "The city is organised around the River Seine, which flows through the historic centre.",
    "Major landmarks include the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral and the Arc de Triomphe. "
    "The city also has extensive rail, metro, bus and regional transport connections.",
    "Paris has a dense urban centre, many neighbourhoods with distinct identities, public gardens and cultural institutions. "
    "Visitors often explore the city on foot because many central landmarks are relatively close together.",
]


def make_context_prompt(repetitions):
    context = "\n\n".join(NOTES[i % len(NOTES)] for i in range(repetitions))
    return (
        "Read the following notes. Based only on these notes, write a concise four-sentence "
        "overview of Paris and its landmarks.\n\n"
        f"NOTES:\n{context}"
    )


prompt_length_rows = []
for label, repetitions in [("Short", 1), ("Medium", 8), ("Long", 32)]:
    prompt = make_context_prompt(repetitions)
    # Two measurements reduce the effect of one unusually fast or slow request.
    for repetition in range(2):
        result = ask(prompt, temperature=0, max_tokens=80)
        prompt_length_rows.append({
            "prompt_size": label,
            "run": repetition + 1,
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "TTFT_ms": result["ttft_ms"],
            "client_latency_ms": result["client_latency_ms"],
        })

prompt_length_df = pd.DataFrame(prompt_length_rows)
display(prompt_length_df)

prompt_length_summary = (
    prompt_length_df.groupby("prompt_size", sort=False)
    .agg(input_tokens=("input_tokens", "median"),
         TTFT_ms=("TTFT_ms", "median"),
         client_latency_ms=("client_latency_ms", "median"))
    .reset_index()
)
display(prompt_length_summary)

if prompt_length_summary["TTFT_ms"].notna().any():
    ax = prompt_length_summary.plot(
        x="input_tokens", y="TTFT_ms", marker="o", legend=False,
        title="Prompt length and server-side TTFT", xlabel="Input tokens", ylabel="Median TTFT (ms)"
    )
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("Server-side TTFT is unavailable. Start vLLM with --enable-per-request-metrics to collect it.")

**Questions — prompt length**

1. What happened to the actual number of input tokens as the prompt grew?
2. How did TTFT change? Is the trend perfectly linear in your measurements?
3. Why is the input processed in parallel during prefill, even though the model is autoregressive?

> Compare medians, not just one run. Small differences can be caused by scheduling, caching, network overhead and measurement noise.

### 2B. Increase output budget — observe decode

Now the input prompt stays fixed, while `max_tokens` changes. This parameter is a **maximum generation budget**, not a promise that the model will use every token: the model may stop earlier when it emits an end-of-turn token.

The prompt asks for a progressively detailed explanation so that larger budgets can be used. Inspect the *actual* output-token count as well as the latency.

In [ ]:
OUTPUT_PROMPT = (
    "Explain how a KV cache helps an autoregressive Transformer generate a long answer. "
    "Include what is stored, what is reused at each decode step, and the memory trade-off. "
    "Give a detailed explanation rather than stopping after one sentence."
)

output_rows = []
for budget in [16, 64, 128]:
    result = ask(OUTPUT_PROMPT, temperature=0, max_tokens=budget)
    output_rows.append({
        "max_tokens_budget": budget,
        "actual_output_tokens": result["output_tokens"],
        "TTFT_ms": result["ttft_ms"],
        "mean_ITL_ms": result["mean_itl_ms"],
        "client_latency_ms": result["client_latency_ms"],
    })

output_df = pd.DataFrame(output_rows)
display(output_df)

ax = output_df.plot(
    x="actual_output_tokens", y="client_latency_ms", marker="o", legend=False,
    title="Actual output length and client-observed latency",
    xlabel="Generated output tokens", ylabel="Client latency (ms)"
)
ax.grid(True, alpha=0.3)
plt.show()

**Questions — output length**

1. Did the model generate exactly the maximum number of tokens each time?
2. What happened to total latency when more tokens were generated?
3. During decode, why does the model generate tokens sequentially?
4. What does the KV cache avoid recomputing, and what memory cost does it introduce?

**Key idea:** prefill processes the prompt and builds the KV cache. Decode then generates tokens sequentially, reusing cached keys and values. More output tokens usually mean more decode steps and therefore a longer response.

**Terminology:** TTFT is the time to the first generated token. ITL is the interval between streamed outputs. TPOT is often reported as an average per-output-token time after the first token. The per-request values shown here are supplied by vLLM when enabled.

---

## 3. Multiple users: latency vs throughput (9 min)

**Objective:** observe why an inference engine needs batching and scheduling.

We send the same type of request several times. First, only one request may be active at a time. Then, up to eight requests may be active concurrently. The model, prompt structure and output budget remain fixed.

The helper function manages the requests; you only change the concurrency value. For this small classroom experiment, we use 16 requests per scenario.

In [ ]:
N_REQUESTS = 16  # Keep this fixed when comparing scenarios.

load_results = {}
load_summaries = []

for concurrency in [1, 8]:
    print(f"Running {N_REQUESTS} requests with concurrency={concurrency} ...")
    requests_df, summary = run_load_test(
        num_requests=N_REQUESTS,
        concurrency=concurrency,
        max_tokens=80,
    )
    load_results[concurrency] = requests_df
    load_summaries.append(summary)

load_summary_df = pd.DataFrame(load_summaries)
display(load_summary_df)

In [ ]:
# Compare system throughput and user-facing latency.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(load_summary_df["concurrency"].astype(str), load_summary_df["output_tokens_per_s"])
axes[0].set_title("Aggregate output throughput")
axes[0].set_xlabel("Maximum concurrency")
axes[0].set_ylabel("Generated tokens / second")
axes[0].grid(axis="y", alpha=0.3)

axes[1].plot(load_summary_df["concurrency"].astype(str), load_summary_df["p50_client_latency_s"], marker="o", label="p50")
axes[1].plot(load_summary_df["concurrency"].astype(str), load_summary_df["p95_client_latency_s"], marker="o", label="p95")
axes[1].set_title("Per-request client latency")
axes[1].set_xlabel("Maximum concurrency")
axes[1].set_ylabel("Latency (seconds)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show the server-side timing metrics when available.
server_metrics_cols = ["concurrency", "p50_ttft_ms", "p95_ttft_ms", "p50_mean_itl_ms"]
display(load_summary_df[server_metrics_cols])

### Questions — serving under load

1. Did output throughput increase when concurrency increased?
2. Did p50 or p95 request latency increase, decrease or stay similar?
3. Why can a system serve more tokens per second while an individual user waits longer?
4. Why does the KV cache consume more memory when more requests are active?
5. What might happen if one long prompt arrives while several users are already receiving generated tokens?

**Key idea:** batching can improve GPU utilisation and aggregate throughput, but it does not guarantee lower latency for each user. Continuous batching lets the engine admit new requests as other sequences finish, rather than waiting for an entire fixed batch to complete. KV cache capacity and scheduling are central constraints.

**Interpret your own numbers.** There is no required direction for every metric: results depend on the model, server settings, input/output lengths and the hardware configuration. If another group is using the same endpoint at the same time, your measurements are not an isolated comparison.

---

## 4. Memory and quantization (3 min)

**Objective:** connect numerical precision to the memory required to store model weights.

For a model with 8 billion parameters, the ideal storage for weights is:

\[
\text{Weight memory (bytes)} = \text{number of parameters} \times \frac{\text{bits per parameter}}{8}
\]

This first-order estimate excludes metadata, KV cache, temporary activations, workspaces and runtime overhead.

In [ ]:
parameters_billions = 8
precisions = [
    {"precision": "BF16 / FP16", "bits_per_parameter": 16},
    {"precision": "INT8", "bits_per_parameter": 8},
    {"precision": "INT4", "bits_per_parameter": 4},
]

memory_rows = []
for item in precisions:
    # Decimal GB: 1 GB = 1e9 bytes.
    memory_gb = parameters_billions * item["bits_per_parameter"] / 8
    memory_rows.append({
        "precision": item["precision"],
        "bits per parameter": item["bits_per_parameter"],
        "ideal weight memory (GB)": memory_gb,
    })

memory_df = pd.DataFrame(memory_rows)
display(memory_df)

### Questions — memory

1. How much ideal memory do the 8B weights occupy in BF16? In INT4?
2. Does INT4 mean the *whole inference process* uses one quarter of the memory?
3. Why can quantization reduce memory without guaranteeing a proportional speed-up?

**Key idea:** quantization reduces the number of bits used to represent values. The KV cache and runtime still need memory. Actual speed depends on the GPU, kernels, model format and workload.

**Note:** distillation and pruning are different approaches. Distillation trains a smaller student model; pruning removes weights or structures, but useful memory and speed gains depend on the sparsity format and implementation. MoE is an architecture in which only selected experts are active for a token, while all expert weights still need to be stored or made available.

---

## 5. Wrap-up (2 min)

Answer these without rerunning the notebook:

1. **Generation:** what is the difference between greedy decoding and sampling?
2. **Latency:** which phase mainly determines the time before the first token, and which phase generates subsequent tokens?
3. **Serving:** why can higher concurrency improve total throughput but worsen per-user latency?

### One-sentence takeaway

Complete this sentence:

> An LLM inference service must balance **__________**, **__________** and **__________** while preserving answer quality.

### What to remember

- Generation is autoregressive: decode produces tokens one after another.
- Prefill processes the prompt and builds the KV cache.
- TTFT, ITL / TPOT, end-to-end latency and throughput describe different aspects of performance.
- KV caching trades additional memory for less repeated computation.
- Batching and scheduling affect how efficiently a GPU serves multiple users.
- Quantization can reduce weight memory, but it is not a guarantee of a proportional latency improvement.

## Further reading

These references correspond to the topics covered in the course presentation:

- [Hugging Face — Text generation strategies](https://huggingface.co/docs/transformers/en/generation_strategies)
- [Hugging Face — KV cache](https://huggingface.co/docs/transformers/en/kv_cache)
- [vLLM — Per-request metrics](https://docs.vllm.ai/en/latest/features/per_request_metrics/)
- [vLLM — Production metrics](https://docs.vllm.ai/en/latest/usage/metrics/)
- [vLLM — `bench serve`](https://docs.vllm.ai/en/latest/cli/bench/serve/)

---

**Instructor note:** this notebook deliberately avoids implementing tokenization, attention, KV-cache allocation or a benchmark client from scratch. Those components are provided by the inference engine so that the practical session can focus on interpreting generation and serving behaviour.